# 경제 분석 및 예측과 데이터 지능 실습5: 데이터 엔지니어링 응용

- 다양한 형식 읽기: `read_json`, `read_xml`, 중첩 구조의 `json_normalize`
- 테이블 결합: `merge`의 `how`/`on`/`validate`/`indicator`
- 고급 그룹 연산: `transform`(그룹 상대값) vs `agg` vs `filter`, 인덱스 정리(`set_index`/`reset_index`)
- 그룹 내 순위와 상위 N: `groupby().rank()`, 그룹별 `idxmax`
- 보고용 구조 변환: `pivot_table(margins)`, `crosstab(normalize)`
- 정규표현식 문자열 추출: `str.extract`
- 날짜·기간 계산: `datetime` 차이로 소요일 만들기, `resample`·`shift`로 시간 단위 집계
- 텍스트 단어 수: `str.split`으로 단어 수를 세고 그룹별로 비교

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
DE = "../datasets/de/"

## 1) 다양한 형식 읽기 — JSON / XML

API 응답은 보통 **JSON**, 공공데이터는 **XML**인 경우가 많습니다.

- `pd.read_json`: 열 지향 JSON을 바로 데이터프레임으로.
- `pd.read_xml`: XML의 반복 요소(`<row>`)를 행으로. `lxml`이 없으면 `parser="etree"`(파이썬 기본 파서)를 쓰면 됩니다.
- 빈 태그(`<f1/>`)는 결측(`NaN`)으로 들어옵니다 — CSV의 빈 칸과 동일하게 다루면 됩니다.

In [2]:
js = pd.read_json(DE + "basic1_json.json")
xm = pd.read_xml(DE + "basic1_xml.xml", parser="etree")

# XML에는 <index> 요소가 그대로 열로 들어오므로 정리
xm = xm.drop(columns=["index"])

print("JSON shape:", js.shape, "| XML shape:", xm.shape)
print("XML의 f1 결측 수:", int(xm["f1"].isna().sum()))   # 빈 태그 → NaN
# 세 형식이 동일한 데이터인지 확인
csv = pd.read_csv(DE + "basic1.csv")
print("JSON == CSV ? ", js.shape == csv.shape and (js["id"] == csv["id"]).all())
js.head(3)

JSON shape: (100, 8) | XML shape: (100, 8)
XML의 f1 결측 수: 31
JSON == CSV ?  True


,id,age,city,f1,f2,f3,f4,f5
0,id01,2.0,서울,NaN,0,NaN,ENFJ,91.297791
1,id02,9.0,서울,70.0,1,NaN,ENFJ,60.339826
2,id03,27.0,서울,61.0,1,NaN,ISTJ,17.252986


**중첩 JSON 다루기.** API 응답은 종종 딕셔너리 안에 딕셔너리가 들어 있습니다.
이때는 `pd.json_normalize`로 평탄화합니다. `sep`으로 중첩 키를 이어 붙입니다.

In [3]:
nested = [
    {"id": 1, "user": {"name": "Kim", "addr": {"city": "Seoul"}}, "amount": 1000},
    {"id": 2, "user": {"name": "Lee", "addr": {"city": "Busan"}}, "amount": 2500},
]
flat = pd.json_normalize(nested, sep="_")
display(flat)

,id,amount,user_name,user_addr_city
0,1,1000,Kim,Seoul
1,2,2500,Lee,Busan


## 2) 테이블 결합 — `merge`

서로 다른 표를 **공통 키**로 이어 붙입니다. SQL의 JOIN과 같습니다.

- `how`: `inner`(교집합) · `left`(왼쪽 보존) · `right` · `outer`(합집합)
- `on`: 양쪽에 같은 이름의 키가 있을 때. 이름이 다르면 `left_on`/`right_on`.
- `validate="many_to_one"`: 관계를 강제 검증 — 의도치 않은 **행 폭증(중복 키)** 을 조기에 잡아줍니다.
- `indicator=True`: 각 행이 어디서 왔는지(`both`/`left_only`/`right_only`) 표시.

`basic3`은 성격유형(`f4`)별 보조 정보(`r1`, `r2`)를 담은 **참조 테이블**입니다.
`basic1`의 각 사람에게 이 정보를 붙입니다(다대일 결합).

In [4]:
b1 = pd.read_csv(DE + "basic1.csv")
b3 = pd.read_csv(DE + "basic3.csv")
print("basic1:", b1.shape, "| basic3(참조):", b3.shape)

merged = b1.merge(b3, on="f4", how="left", validate="many_to_one", indicator=True)
print("결합 결과:", merged.shape)
print("매칭 상태:\n", merged["_merge"].value_counts())
display(merged[["id", "f4", "r1", "r2"]].head())

basic1: (100, 8) | basic3(참조): (16, 3)
결합 결과: (100, 11)
매칭 상태:
 _merge
both          100
left_only       0
right_only      0
Name: count, dtype: int64


,id,f4,r1,r2
0,id01,ENFJ,INFP,ISFP
1,id02,ENFJ,INFP,ISFP
2,id03,ISTJ,ESFP,NaN
3,id04,INFP,ENFJ,ENTJ
4,id05,ISFJ,ESFP,ESTP


## 3) 그룹 연산 응용 — `transform` vs `agg` vs `filter`

`groupby` 뒤에 무엇을 붙이느냐로 결과 모양이 달라집니다.

| 메서드 | 결과 행 수 | 용도 |
|--------|-----------|------|
| `agg` | 그룹 수만큼 **축소** | 그룹 요약표 |
| `transform` | **원래 행 수 유지** | 그룹 통계량을 각 행에 되돌림 → 그룹 상대값 |
| `filter` | 조건 통과 그룹의 행만 | 그룹 단위 선별 |

`purchase`에서 **자기 세그먼트 평균 대비** 얼마나 높/낮게 샀는지(그룹 내 z-score)를 `transform`으로 만듭니다.

In [5]:
pu = pd.read_csv(DE + "purchase.csv")
print("세그먼트:", pu["세그먼트"].unique().tolist())
print("카테고리:", pu["카테고리"].unique().tolist())

# (agg) 세그먼트 × 카테고리 요약
summary = pu.groupby(["세그먼트", "카테고리"], as_index=False).agg(
    n=("구매금액", "size"),
    mean_amt=("구매금액", "mean"),
    max_amt=("구매금액", "max"),
)
print("\n[agg: 그룹 요약]")
display(summary.head())

# (transform) 그룹 내 z-score: 같은 세그먼트 안에서의 상대 위치
pu["amt_z_in_seg"] = pu.groupby("세그먼트")["구매금액"].transform(lambda s: (s - s.mean()) / s.std())
print("[transform: 세그먼트 내 z-score]")
display(pu[["고객ID", "세그먼트", "구매금액", "amt_z_in_seg"]].head())

세그먼트: ['일반', '프리미엄']
카테고리: ['생활용품', '식품', '패션', '가전']

[agg: 그룹 요약]


,세그먼트,카테고리,n,mean_amt,max_amt
0,일반,가전,7,29051.571429,43810
1,일반,생활용품,10,23000.300000,41487
2,일반,식품,16,31309.937500,47918
3,일반,패션,12,33189.333333,43413
4,프리미엄,가전,14,33072.857143,49482


[transform: 세그먼트 내 z-score]


,고객ID,세그먼트,구매금액,amt_z_in_seg
0,C100,일반,18807,-0.852473
1,C101,일반,47348,1.399048
2,C102,프리미엄,31432,0.099531
3,C103,일반,22749,-0.541500
4,C104,프리미엄,46459,1.259560


예시: 각 고객의 구매금액을 *자기 세그먼트* 기준으로 표준화(z-score)했을 때,
z-score가 **1보다 큰**(자기 세그먼트 평균보다 1표준편차 이상 많이 산) 고객 수를 구하라.

In [6]:
print(int((pu["amt_z_in_seg"] > 1).sum()))

19


### 그룹 결과의 인덱스 정리 (`set_index` / `reset_index`)

위에서 `as_index=False`를 줬는데, 이게 무슨 뜻일까요? 여러 키로 `groupby`하면 그 키들이 결과의 **인덱스(MultiIndex)** 로 들어갑니다.

- `reset_index()`: 인덱스를 다시 보통 열로 되돌립니다. `as_index=False`는 바로 이 과정을 한 번에 해 주는 단축형입니다.
- `set_index(열)`: 반대로 특정 열을 인덱스로 올립니다.

그룹 결과를 평평한 표로 정리해 두면, 바로 다음에 배울 **순위·피벗·병합**에 그대로 이어 쓸 수 있습니다.

In [7]:
multi = pu.groupby(["세그먼트", "연령대"])["구매금액"].mean()   # 키 2개 → MultiIndex
print("결과 인덱스 이름:", multi.index.names)

flat = multi.reset_index()                                      # 인덱스를 열로 되돌림
same = flat.equals(pu.groupby(["세그먼트", "연령대"], as_index=False)["구매금액"].mean())
print("reset_index() 결과가 as_index=False와 같은가:", same)
display(flat.head(3))

결과 인덱스 이름: ['세그먼트', '연령대']
reset_index() 결과가 as_index=False와 같은가: True


,세그먼트,연령대,구매금액
0,일반,10대,35124.700000
1,일반,20대,33159.142857
2,일반,30대,22955.250000


## 4) 그룹 내 순위와 상위 N

"카테고리별 구매금액 상위 2명"처럼 **그룹마다 상위 N개**를 뽑는 작업입니다.

- `groupby(key)[col].rank(ascending=False, method="min")` 으로 그룹 내 순위를 매기고 필터링하거나,
- 정렬 후 `groupby(key).head(n)` 으로 간단히 뽑을 수 있습니다.

In [8]:
# 방법 A: 그룹 내 순위 후 필터
pu["rank_in_cat"] = pu.groupby("카테고리")["구매금액"].rank(ascending=False, method="min")
top2_a = pu[pu["rank_in_cat"] <= 2].sort_values(["카테고리", "rank_in_cat"])

# 방법 B: 정렬 후 head
top2_b = (pu.sort_values("구매금액", ascending=False)
            .groupby("카테고리", as_index=False)
            .head(2))

print("두 방법의 행 수 일치:", len(top2_a) == len(top2_b))
display(top2_a[["카테고리", "고객ID", "구매금액", "rank_in_cat"]])

두 방법의 행 수 일치: True


,카테고리,고객ID,구매금액,rank_in_cat
21,가전,C121,49482,1.0
30,가전,C130,47344,2.0
65,생활용품,C165,46430,1.0
62,생활용품,C162,45821,2.0
61,식품,C161,49597,1.0
31,식품,C131,47918,2.0
56,패션,C156,49472,1.0
33,패션,C133,47956,2.0


## 5) 보고용 구조 변환 — `pivot_table` / `crosstab`

집계 결과를 **행 × 열 매트릭스**로 펼쳐 보고서 형태로 만듭니다.

- `pivot_table(..., margins=True)`: 행/열 합계(총계) 자동 추가.
- `crosstab(..., normalize="index")`: 빈도표를 **행 기준 비율**로 정규화 → 구성비 비교에 유용.

In [9]:
# 연령대 × 카테고리 평균 구매금액 (+ 총계)
pivot = pu.pivot_table(index="연령대", columns="카테고리",
                       values="구매금액", aggfunc="mean", margins=True, margins_name="전체")
print("[연령대 × 카테고리 평균 구매금액]")
display(pivot.round(0))

# 세그먼트별 카테고리 '구성비'
comp = pd.crosstab(pu["세그먼트"], pu["카테고리"], normalize="index").round(3)
print("[세그먼트별 카테고리 구성비(행 합=1)]")
display(comp)

[연령대 × 카테고리 평균 구매금액]


카테고리,가전,생활용품,식품,패션,전체
연령대,,,,,
10대,33920.0,27326.0,28776.0,41584.0,32356.0
20대,30366.0,25607.0,35462.0,31862.0,31545.0
30대,23272.0,25661.0,20214.0,25374.0,23068.0
40대,33868.0,30424.0,39032.0,46270.0,34901.0
50대,31766.0,23388.0,26964.0,27980.0,27143.0
전체,31732.0,26787.0,29028.0,32024.0,29904.0


[세그먼트별 카테고리 구성비(행 합=1)]


카테고리,가전,생활용품,식품,패션
세그먼트,,,,
일반,0.156,0.222,0.356,0.267
프리미엄,0.255,0.200,0.291,0.255


## 6) 정규표현식으로 문자열 추출 — `str.extract`

`Titanic`의 `Name`은 `"성, 호칭. 이름"` 구조입니다. 정규표현식의 **그룹 `( )`** 으로 호칭만 뽑습니다.

- 패턴 `r",\s*([^.]+)\."` → 쉼표 뒤 공백 이후부터 마침표 전까지를 한 그룹으로 캡처.
- 호칭은 성별·결혼·신분 정보를 담고 있어 좋은 파생변수가 됩니다(실습5-2에서 활용 가능).

In [10]:
tt = pd.read_csv(DE + "Titanic.csv")
tt["title"] = tt["Name"].str.extract(r",\s*([^.]+)\.")
print("[호칭 분포]")
display(tt["title"].value_counts())

# 희소 호칭은 'Rare'로 묶기 — 모델 입력에 자주 쓰는 정리 방식
common = ["Mr", "Miss", "Mrs", "Master"]
tt["title_grp"] = tt["title"].where(tt["title"].isin(common), "Rare")
print("[호칭별 생존율]")
display(tt.groupby("title_grp")["Survived"].agg(["count", "mean"]).round(3))

[호칭 분포]


title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64

[호칭별 생존율]


,count,mean
title_grp,,
Master,40,0.575
Miss,182,0.698
Mr,517,0.157
Mrs,125,0.792
Rare,27,0.444


## 7) 날짜·기간 계산

두 시점의 차이로 **소요 기간**을 만드는 작업입니다.
`datetime`끼리 빼면 `Timedelta`가 나오고, `.dt.days`로 일수를 얻습니다.

`e-commerce` 데이터로 **주문 → 도착 배송 소요일**을 계산합니다.

In [11]:
ec = pd.read_csv(DE + "e-commerce.csv")
ec["OrderDate"] = pd.to_datetime(ec["OrderDate"])
ec["ArrivalDate"] = pd.to_datetime(ec["ArrivalDate"])
ec["delivery_days"] = (ec["ArrivalDate"] - ec["OrderDate"]).dt.days

print("[카테고리별 평균 배송 소요일]")
display(ec.groupby("Category")["delivery_days"].mean().round(2).sort_values())
print("전체 평균 배송일:", round(ec["delivery_days"].mean(), 2))

[카테고리별 평균 배송 소요일]


Category
배송     4.00
환경     4.00
품질     4.14
가격     4.50
서비스    5.81
Name: delivery_days, dtype: float64

전체 평균 배송일: 5.03


**미션 2.** `e-commerce`에서 배송 소요일(`delivery_days`)이 **7일을 초과**하는 주문의 비율(%)을
소수 첫째 자리까지 구하라.

In [12]:
rate = (ec["delivery_days"] > 7).mean() * 100
print(round(rate, 1))

30.0


### 날짜를 인덱스로 — `resample` 과 `shift`

날짜 자체를 인덱스로 올리면(`set_index`, 바로 앞에서 배운 것) 시간 단위로 묶는 일이 한 줄로 됩니다.

- `resample("ME")`(월말)·`"W"`(주)·`"D"`(일) 같은 빈도로 다시 묶어 집계합니다.
  연·월을 일일이 뽑아 `groupby`하는 대신, 날짜 인덱스가 그 일을 대신해 줍니다.
- `shift(1)`은 한 시점 전 값을 끌어옵니다. 지금 값에서 빼면 **전월 대비 증감**이 됩니다.

In [13]:
sb = pd.read_csv(DE + "sales_branch.csv")
sb["거래일"] = pd.to_datetime(sb["거래일"])

monthly = sb.set_index("거래일")["매출액"].resample("ME").sum()   # 월말 기준 매출 합계
report = pd.DataFrame({
    "매출합계": monthly.astype(int),
    "전월대비": monthly - monthly.shift(1),                       # 한 달 전과의 차이
})
print("[월별 매출과 전월 대비 증감]")
display(report)

[월별 매출과 전월 대비 증감]


,매출합계,전월대비
거래일,,
2024-01-31,1988732,NaN
2024-02-29,1735446,-253286.0
2024-03-31,1741998,6552.0
2024-04-30,638266,-1103732.0


## 8) 텍스트에서 단어 수 세기

문장을 띄어쓰기로 나눠 **단어 수**를 세는 것은 텍스트 데이터의 기본 피처입니다.

- `str.split()`로 단어 리스트를 만들고 `str.len()`으로 개수를 셉니다.
- 글자 수는 문자열에 `str.len()`을 바로 적용합니다.
- 이렇게 만든 단어 수를 그룹(label)별로 평균 내어 비교할 수 있습니다.

In [14]:
hs = pd.read_csv(DE + "hamspam.csv")
hs["n_words"] = hs["text"].str.split().str.len()   # 띄어쓰기 기준 단어 수
hs["n_chars"] = hs["text"].str.len()               # 글자 수(공백 포함)

print("[label별 평균 단어 수]")
display(hs.groupby("label")["n_words"].mean().round(2))
display(hs[["label", "text", "n_words", "n_chars"]].head(3))

[label별 평균 단어 수]


label
ham     10.07
spam    10.02
Name: n_words, dtype: float64

,label,text,n_words,n_chars
0,ham,Incididunt tempor elit sit do aliqua eiusmod a...,10,66
1,ham,Adipiscing tempor lorem ipsum dolore tempor.,6,44
2,spam,Tempor tempor magna sed incididunt consectetur.,6,47


**미션 3.** 메시지의 평균 단어 수가 더 많은 `label`(ham 또는 spam)을 구하라.

In [15]:
mean_words = hs.groupby("label")["n_words"].mean()
print(mean_words.idxmax())

ham


## 9) 종합문제

데이터를 다루다 보면 *분할 → 그룹 집계 → 병합 → 파생 → 선택*처럼 여러 단계를 이어서 처리하게 됩니다.
이번 실습에서 배운 `groupby`·`transform`·`merge`를 묶어 풉니다.

**종합문제 1.** `purchase`에서
1. 각 고객의 구매금액을 *자기 연령대* 기준으로 표준화한다(그룹 내 z-score).
2. 표준화 점수가 가장 높은 **상위 5명**의 *원래 구매금액 평균*을 정수로 구하라.

In [16]:
cap = pd.read_csv(DE + "purchase.csv")
cap["z_in_age"] = cap.groupby("연령대")["구매금액"].transform(lambda s: (s - s.mean()) / s.std())
top5 = cap.nlargest(5, "z_in_age")
display(top5[["고객ID", "연령대", "구매금액", "z_in_age"]])
print("상위 5명 평균 구매금액:", int(top5["구매금액"].mean()))

,고객ID,연령대,구매금액,z_in_age
61,C161,30대,49597,1.868355
6,C106,50대,45262,1.651385
21,C121,20대,49482,1.492251
89,C189,50대,43088,1.453247
84,C184,50대,42732,1.420801


상위 5명 평균 구매금액: 46032


**종합문제 2.** (분할 → 집계 → 병합 → 차이) `purchase`에서 세그먼트를 둘로 나눠 비교합니다.

1. `세그먼트 == '일반'` 과 `'프리미엄'` 으로 데이터를 나눈다.
2. 각각 **연령대별 평균 구매금액**을 구한다.
3. 두 결과를 `merge`해 연령대별 **평균 구매금액 차이(절댓값)** 를 만들고, **차이가 가장 큰 연령대**를 구하라.

> *분할 → `groupby` → `merge` → 파생 → 정렬/선택* 은 두 집단을 비교할 때 자주 쓰는 흐름입니다.

In [17]:
pu = pd.read_csv(DE + "purchase.csv")
normal = pu[pu["세그먼트"] == "일반"].groupby("연령대")["구매금액"].mean()
premium = pu[pu["세그먼트"] == "프리미엄"].groupby("연령대")["구매금액"].mean()

diff = pd.merge(normal.rename("일반"), premium.rename("프리미엄"), on="연령대")
diff["차이"] = (diff["프리미엄"] - diff["일반"]).abs()
display(diff.round(0))
print("평균 구매금액 차이가 가장 큰 연령대:", diff["차이"].idxmax())

,일반,프리미엄,차이
연령대,,,
10대,35125.0,28401.0,6724.0
20대,33159.0,30838.0,2321.0
30대,22955.0,23197.0,242.0
40대,32803.0,36126.0,3322.0
50대,25844.0,28442.0,2598.0


평균 구매금액 차이가 가장 큰 연령대: 10대


## 생각해보기

1. `merge`에서 `validate="many_to_one"`을 빼고 `basic3`에 중복 `f4`가 있었다면 어떤 문제가 생길까요?
2. `transform`으로 만든 그룹 내 z-score와, `agg`로 만든 그룹 평균을 다시 `merge`하는 방법은 결과가 같을까요? 어느 쪽이 간결한가요?
3. `crosstab(normalize="index")` 대신 `normalize="columns"`로 바꾸면 해석이 어떻게 달라지나요?
4. 호칭(`title`)을 `Rare`로 묶는 기준(빈도 임계값)을 바꾸면 모델 성능에 어떤 영향이 있을까요?
5. 배송 소요일을 *시간 단위*까지 반영하려면(`.dt.days` 대신) 어떻게 계산해야 할까요?